<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.lab.1. Wisconsin Cancer Classification
  </strong>
</div>

### Background

You will continue working with the Wisconsin Diagnostic Cancer Dataset, but this time implementing classification models using scikit-learn.

The dataset includes:
- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- Binary classification: **Malignant** or **Benign** diagnosis

### Description

Your task is to implement logistic regression using scikit-learn to:

- Implement logistic regression using all features
- Find the best performing pair of features for classification
- Compare the accuracy between using all features vs. the best pair
- Visualize the decision boundary for the best feature pair

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **wisconsin-cancer-classification.ipynb**
- The notebook should include visualizations of the decision boundaries
- A clear comparison of performance between using all features vs. the best pair
- Use the provided template as a starting point

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

from sklearn.datasets import load_breast_cancer

In [ ]:
raw_data = load_breast_cancer(as_frame=True)
df = raw_data.data
df.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
from itertools import combinations
from matplotlib.colors import ListedColormap

In [ ]:
plt.style.use("ggplot")
sns.set(font_scale=1.2)
np.random.seed(42)

In [ ]:
print("1. Cargando el conjunto de datos de Cáncer de Wisconsin")
raw_data = load_breast_cancer(as_frame=True)
df = raw_data.data
target = raw_data.target
feature_names = raw_data.feature_names

In [ ]:
print(f"Forma del conjunto de datos: {df.shape}")
print(f"Distribución de clases: {pd.Series(target).value_counts()}")
print(f"Nombres de características: {feature_names}")

In [ ]:
df.head()

In [ ]:
print("\n2. Exploración inicial de datos")

In [ ]:
print("Estadísticas descriptivas:")
df.describe().T

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x=target)
plt.title("Distribución de clases (0: Maligno, 1: Benigno)")
plt.xlabel("Clase")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
print("\n3. Preprocesamiento de datos")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, target, test_size=0.2, random_state=42, stratify=target
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

In [ ]:
print("\n4. Implementación de Regresión Logística con todas las características")
modelo_completo = LogisticRegression(max_iter=1000, random_state=42)
modelo_completo.fit(X_train, y_train)

In [ ]:
y_pred_completo = modelo_completo.predict(X_test)
accuracy_completo = accuracy_score(y_test, y_pred_completo)

In [ ]:
print(f"Precisión del modelo con todas las características: {accuracy_completo:.4f}")
print("\nMatriz de confusión:")
cm_completo = confusion_matrix(y_test, y_pred_completo)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_completo,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Maligno", "Benigno"],
    yticklabels=["Maligno", "Benigno"],
)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Modelo Completo")
plt.show()

In [ ]:
print("\nInforme de clasificación:")
print(
    classification_report(y_test, y_pred_completo, target_names=["Maligno", "Benigno"])
)

In [ ]:
cv_scores_completo = cross_val_score(modelo_completo, X_scaled, target, cv=5)
print(
    f"Precisión promedio de validación cruzada (5-fold): {cv_scores_completo.mean():.4f} ± {cv_scores_completo.std():.4f}"
)

In [ ]:
print("\n5. Búsqueda del mejor par de características")


def evaluar_par_caracteristicas(X, y, idx1, idx2):
    X_par = X[:, [idx1, idx2]]

    X_train_par, X_test_par, y_train_par, y_test_par = train_test_split(
        X_par, y, test_size=0.2, random_state=42, stratify=y
    )

    modelo = LogisticRegression(max_iter=1000, random_state=42)
    modelo.fit(X_train_par, y_train_par)

    y_pred = modelo.predict(X_test_par)
    accuracy = accuracy_score(y_test_par, y_pred)

    return accuracy, modelo, (X_train_par, X_test_par, y_train_par, y_test_par)

In [ ]:
selector = SelectKBest(f_classif, k=10)
X_new = selector.fit_transform(X_scaled, target)
top_indices = np.argsort(selector.scores_)[-10:]
print("Top 10 características más importantes:")
for idx in top_indices:
    print(f"- {feature_names[idx]}: {selector.scores_[idx]:.4f}")

In [ ]:
mejores_pares = []
for idx1, idx2 in combinations(top_indices, 2):
    accuracy, modelo, datos = evaluar_par_caracteristicas(X_scaled, target, idx1, idx2)
    mejores_pares.append((accuracy, idx1, idx2, modelo, datos))
mejores_pares.sort(reverse=True)

In [ ]:
print("\nLos 5 mejores pares de características:")
for i, (acc, idx1, idx2, _, _) in enumerate(mejores_pares[:5]):
    print(f"{i+1}. {feature_names[idx1]} y {feature_names[idx2]}: {acc:.4f}")

In [ ]:
mejor_accuracy, mejor_idx1, mejor_idx2, mejor_modelo, mejores_datos = mejores_pares[0]
mejor_X_train, mejor_X_test, mejor_y_train, mejor_y_test = mejores_datos

print(
    f"\nMejor par de características: {feature_names[mejor_idx1]} y {feature_names[mejor_idx2]}"
)
print(f"Precisión con el mejor par: {mejor_accuracy:.4f}")

In [ ]:
print("\n6. Comparación de modelos: Todas las características vs. Mejor par")

comparacion = pd.DataFrame(
    {
        "Modelo": ["Todas las características", "Mejor par de características"],
        "Características": [
            f"Todas ({df.shape[1]})",
            f"{feature_names[mejor_idx1]} y {feature_names[mejor_idx2]}",
        ],
        "Precisión": [accuracy_completo, mejor_accuracy],
    }
)

print(comparacion)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x="Modelo", y="Precisión", data=comparacion)
plt.title("Comparación de precisión: Todas las características vs. Mejor par")
plt.ylim(0.8, 1.0)
plt.show()

In [ ]:
print("\n7. Visualización de la frontera de decisión para el mejor par")


def plot_decision_boundary(X, y, modelo, feature_names):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    cmap_light = ListedColormap(["#FFAAAA", "#AAFFAA"])
    cmap_bold = ListedColormap(["#FF0000", "#00FF00"])

    plt.figure(figsize=(12, 10))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor="k", s=50)
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title(f"Frontera de Decisión con {feature_names[0]} y {feature_names[1]}")
    plt.tight_layout()
    plt.show()

In [ ]:
X_mejor_par = X_scaled[:, [mejor_idx1, mejor_idx2]]
nombre_mejor_par = [feature_names[mejor_idx1], feature_names[mejor_idx2]]

plot_decision_boundary(X_mejor_par, target, mejor_modelo, nombre_mejor_par)

In [ ]:
print("\n8. Análisis de coeficientes de la regresión logística")

coef_completo = pd.DataFrame(
    {"Característica": feature_names, "Coeficiente": modelo_completo.coef_[0]}
)
coef_completo = coef_completo.sort_values(by="Coeficiente", ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(x="Coeficiente", y="Característica", data=coef_completo.head(10))
plt.title("Top 10 Características con mayor coeficiente (modelo completo)")
plt.tight_layout()
plt.show()

In [ ]:
coef_par = pd.DataFrame(
    {
        "Característica": [feature_names[mejor_idx1], feature_names[mejor_idx2]],
        "Coeficiente": mejor_modelo.coef_[0],
    }
)

plt.figure(figsize=(10, 6))
sns.barplot(x="Característica", y="Coeficiente", data=coef_par)
plt.title("Coeficientes para el mejor par de características")
plt.tight_layout()
plt.show()

El análisis de clasificación utilizando regresión logística sobre el conjunto de datos de Cáncer de Wisconsin reveló varios hallazgos importantes:

    Composición del conjunto de datos: Se analizaron 569 instancias con 30 características, con una distribución de clases de 357 casos benignos (62.7%) y 212 casos malignos (37.3%).

    Rendimiento del modelo completo:
La regresión logística utilizando todas las 30 características logró una precisión muy alta de 98.25% en el conjunto de prueba.
La validación cruzada de 5 divisiones confirmó la robustez del modelo, con una precisión promedio de 98.07% ± 0.65%.
Las métricas de clasificación mostraron un equilibrio excelente, con precisión y recall de aproximadamente 98% para tumores malignos y 99% para benignos.

    Características más importantes: El análisis de selectividad identificó las 10 características más discriminativas, siendo las tres principales:
"worst concave points" (F-score: 964.39)
"worst perimeter" (F-score: 897.94)
"mean concave points" (F-score: 861.68)

    Mejor par de características: De entre todas las combinaciones posibles de las 10 mejores características:
La combinación de "mean concavity" y "worst area" resultó ser la más efectiva, logrando una precisión del 96.49%.
Esta precisión es solo 1.75 puntos porcentuales menor que el modelo que utiliza las 30 características.

    Visualización de la frontera de decisión: El gráfico de la frontera de decisión para el mejor par de características mostró una clara separación entre las clases maligna y benigna, con relativamente pocos casos en la zona de transición.

    Análisis de coeficientes: Los coeficientes del modelo logístico revelaron qué características tienen mayor influencia en la clasificación, tanto en el modelo completo como en el modelo reducido con solo dos características.
